# Module 1: `tensor.py` — Reverse-Mode Autograd Engine & Broadcasting VJP

This notebook walks through the core `Tensor` computation graph, topological sorting (`backward`), broadcasting gradient reduction (`_unbroadcast`), and centered finite-difference verification (`numerical_grad_check`).

## 1. Imports & Path Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import numpy as np
from tensorgrad.tensor import Tensor, _unbroadcast, numerical_grad_check
print('Loaded tensorgrad.tensor successfully.')

Loaded tensorgrad.tensor successfully.


## 2. Multi-Path DAG Accumulation & Topological Order
When a node participates along multiple paths ($a = 3x,\; b = a + 1,\; L = a \cdot b$), gradients accumulate via `+=` in reverse topological order.

In [2]:
x = Tensor([2.0])
a = x * 3.0
b = a + 1.0
L = a * b
L.backward()
print(f'Forward: x={x.data[0]}, a={a.data[0]}, b={b.data[0]}, L={L.data[0]}')
print(f'Backward analytic dL/dx = {x.grad[0]} (Expected: 18*x + 3 = 39.0)')

Forward: x=2.0, a=6.0, b=7.0, L=42.0
Backward analytic dL/dx = 39.0 (Expected: 18*x + 3 = 39.0)


## 3. Broadcasting VJP (`_unbroadcast`)
Adding a bias vector `b` of shape `(3,)` to a batch matrix `X` of shape `(4, 3)` replicates `b` across 4 rows during the forward pass and sums gradients along `axis=0` during the backward pass.

In [3]:
X = Tensor(np.arange(12, dtype=np.float64).reshape(4, 3))
b = Tensor([1.0, -2.0, 0.5])
loss = ((X - b) * 2.0).sum()
loss.backward()
print('X.grad[:2] (first 2 rows):\n', X.grad[:2])
print('b.grad (summed over 4 rows):', b.grad)

X.grad[:2] (first 2 rows):
 [[2. 2. 2.]
 [2. 2. 2.]]
b.grad (summed over 4 rows): [-8. -8. -8.]


## 4. Centered Finite-Difference Gradient Check
We verify analytic gradients against centered finite differences with step $\epsilon = 10^{-5}$.

In [4]:
x_sample = Tensor([[0.5, -0.7, 1.2], [0.3, 0.9, -0.4]])
chk = numerical_grad_check(
    lambda t: ((t ** 2) * t.exp() + t.relu() + t.tanh()).mean(),
    x_sample,
)
print(f"Gradcheck passed: {chk['passed']} | Relative error: {chk['rel_error']:.3e}")
print('Analytic grad[0]:', np.round(chk['analytic_grad'][0], 6))
print('Numeric  grad[0]:', np.round(chk['numerical_grad'][0], 6))

Gradcheck passed: True | Relative error: 6.413e-11
Analytic grad[0]: [0.641225 0.030474 2.342378]
Numeric  grad[0]: [0.641225 0.030474 2.342378]


## Conclusion
`Tensor` accurately tracks multi-path DAG topologies, reverses NumPy broadcasting rules via `_unbroadcast`, and matches centered finite-difference derivatives to $\sim 10^{-11}$ relative error.